# 05 · Sub-national escalation warning, threat scorecard and what-if scenarios
**Target:** an area *escalates* next month if its deaths are ≥ 10 **and** ≥ 1.5 × its trailing 3-month mean.
That is about 2–4% of area-months, a rare-event problem, so we report ROC-AUC, PR-AUC, Brier score and the
operational **precision of a top-10 watch-list** each month.

Folds: train on target months up to Dec *Y − 1*, test on Jan–Sep *Y*, for *Y* = 2023–2026. Features and
hyper-parameters were chosen on 2023–2025; 2026 is the withheld year.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from src import config as C, viz

In [2]:
from src import models as M
am = pd.read_parquet(C.PROCESSED / "admin1_month.parquet")
cm = pd.read_parquet(C.PROCESSED / "case_month.parquet")
ew = M.early_warning_all(am)
ew["report"].round(3)

,model,year,ROC_AUC,PR_AUC,Brier,precision_at_10,recall_at_20,base_rate,n,positives
0,LightGBM classifier,2023,0.909,0.254,0.020,0.275,0.653,0.024,2664,65
1,Logistic regression (5 signals),2023,0.904,0.209,0.027,0.242,0.675,0.024,2664,65
2,Rule: rank by last month's deaths,2023,0.847,0.227,0.026,0.217,0.665,0.024,2664,65
3,LightGBM classifier,2024,0.845,0.129,0.023,0.183,0.464,0.024,2664,65
4,Logistic regression (5 signals),2024,0.827,0.123,0.027,0.100,0.422,0.024,2664,65
5,Rule: rank by last month's deaths,2024,0.732,0.096,0.035,0.117,0.466,0.024,2664,65
6,LightGBM classifier,2025,0.804,0.240,0.024,0.267,0.651,0.028,2664,74
7,Logistic regression (5 signals),2025,0.816,0.134,0.027,0.133,0.634,0.028,2664,74
8,Rule: rank by last month's deaths,2025,0.785,0.119,0.036,0.125,0.561,0.028,2664,74
9,LightGBM classifier,2026,0.795,0.133,0.038,0.144,0.327,0.041,1998,82


In [3]:
viz.fig_ew_curves(ew['preds'], fold=2026)

In [4]:
rep = ew["report"]; r26 = rep[rep.year == 2026].set_index("model")
lift = r26.precision_at_10 / r26.base_rate
lift.round(1).rename("lift of top-10 watch-list over random (2026)").to_frame()

,lift of top-10 watch-list over random (2026)
model,
LightGBM classifier,3.500
Logistic regression (5 signals),1.600
Rule: rank by last month's deaths,1.100


In [5]:
imp = M.shap_importance(ew["models"][2026], ew["preds"][ew["preds"].fold == 2026][M.EW_FEATURES])
viz.fig_shap(imp, title="What drives the escalation warning (mean |SHAP|)")

**Reading.** On the withheld 2026 months the classifier ranks escalating areas with ROC-AUC ≈ 0.80, against
0.73 for the rule-of-thumb 'watch last month's deadliest areas'. Its top-10 list is about **3.5× better than random**.
The strongest signals are the area's own smoothed intensity, the *whole conflict's* tempo and how many areas are
active (spread), and the country's momentum: escalation is contagious within a theatre.

**So what.** This is an **I&W tasking tool**: it ranks where to point ISR and analysts next month. It is not a
prediction of decisions to go to war.

## Live watch-list (scored on Sep 2026, for Oct 2026)

In [6]:
live = ew["live"]
live[~live.admin1.str.startswith("(")].head(15)[["conflict_case", "country_iso3", "admin1", "deaths", "base3",
                                                 "momentum_3v12", "p_escalation"]].round(3)

,conflict_case,country_iso3,admin1,deaths,base3,momentum_3v12,p_escalation
2447,Afghanistan–Pakistan,AFG,Kandahar province,10.000,3.333,-0.657,0.232
5813,Afghanistan–Pakistan,PAK,Balochistan province,92.000,74.000,1.153,0.214
2906,Afghanistan–Pakistan,AFG,Kunar province,40.000,13.333,-0.105,0.212
22337,US–Iran,IRN,Hormozgan province,9.000,12.071,0.522,0.164
15911,Russia–Ukraine,UKR,Chernihiv oblast,4.000,3.333,-0.127,0.148
9332,Israel–Hamas,PSE,Gaza Strip,128.062,133.597,-0.013,0.142
16217,Russia–Ukraine,UKR,Dnipropetrovsk oblast,52.500,47.833,-0.084,0.140
18665,Russia–Ukraine,UKR,Sumy oblast,31.000,31.667,-0.240,0.138
18053,Russia–Ukraine,UKR,Odessa oblast,11.000,16.667,0.742,0.136
1835,Afghanistan–Pakistan,AFG,Hilmand province,10.000,3.333,0.346,0.135


## Threat scorecard (RAG)

In [7]:
from src import insights
fut = pd.read_parquet(C.PROCESSED / "forecast_future.parquet")
th = insights.theatre_spread(insights.load("events"))
rag = M.rag_scorecard(cm, fut, live, th)
display(rag.round(2))
viz.fig_rag(rag)

,conflict_case,rating,score,forecast_next,pi80_lo,pi80_hi,forecast_month,last6_mean,trend_ratio,hotspots,peak_last12,top_areas,countries_this_year,countries_last_year,pts_intensity,pts_trend,pts_hotspots,pts_spread,pts_volatility
0,Russia–Ukraine,RED,3,"6,780.410","3,315.960","16,707.680",2026-10-01,"6,357.930",1.070,0,"10,767.890",Chernihiv (15%); Dnipropetrovsk (14%); Sumy (14%),2,2,2,0,0,0,1
1,Israel–Hamas,GREEN,1,128.070,62.130,317.010,2026-10-01,134.000,0.960,0,663.380,Gaza Strip (14%); West Bank (11%); Haifa distr...,3,4,1,0,0,0,0
2,Afghanistan–Pakistan,AMBER,3,318.350,155.200,785.840,2026-10-01,284.080,1.120,3,540.510,Kandahar (23%); Balochistan (21%); Kunar (21%),2,2,1,0,2,0,0
3,US–Iran,AMBER,2,26.270,12.340,66.200,2026-10-01,120.810,0.220,1,"2,825.890",Hormozgan (16%); Khuzestan (12%); Kermanshah (9%),14,4,0,-1,1,1,1


Rules (transparent, from model outputs only): **intensity** of the next-month forecast, **trend** against
the last 6 months, number of **hotspot** areas with escalation probability ≥ 15%, **spread** to more countries
than last year, and **volatility** (a ≥ 1,000-death month in the last year). RED = active high-intensity war or ≥ 5
points; AMBER = 2–4; GREEN ≤ 1.

## What-if scenario: 'If Af–Pak violence rises 30% for three months, what does the model say?'

In [8]:
import joblib
art = joblib.load(C.ROOT / "models" / "models.joblib")
q = art["conformal"]
for shock in (0, 30, 100):
    sc = M.scenario_forecast(cm, "Afghanistan–Pakistan", shock, 3, art["m_case_full"], art["members"], q)
    print(f"shock {shock:+d}%:", sc[["target_month", "pred", "pi80_lo", "pi80_hi"]].round(0).to_dict("records"))

shock +0%: [{'target_month': Timestamp('2026-10-01 00:00:00'), 'pred': 318.0, 'pi80_lo': 155.0, 'pi80_hi': 786.0}, {'target_month': Timestamp('2026-11-01 00:00:00'), 'pred': 316.0, 'pi80_lo': 143.0, 'pi80_hi': 1428.0}, {'target_month': Timestamp('2026-12-01 00:00:00'), 'pred': 291.0, 'pi80_lo': 132.0, 'pi80_hi': 1313.0}]
shock +30%: [{'target_month': Timestamp('2026-10-01 00:00:00'), 'pred': 359.0, 'pi80_lo': 175.0, 'pi80_hi': 887.0}, {'target_month': Timestamp('2026-11-01 00:00:00'), 'pred': 359.0, 'pi80_lo': 162.0, 'pi80_hi': 1617.0}, {'target_month': Timestamp('2026-12-01 00:00:00'), 'pred': 330.0, 'pi80_lo': 149.0, 'pi80_hi': 1490.0}]
shock +100%: [{'target_month': Timestamp('2026-10-01 00:00:00'), 'pred': 469.0, 'pi80_lo': 229.0, 'pi80_hi': 1158.0}, {'target_month': Timestamp('2026-11-01 00:00:00'), 'pred': 473.0, 'pi80_lo': 214.0, 'pi80_hi': 2134.0}, {'target_month': Timestamp('2026-12-01 00:00:00'), 'pred': 452.0, 'pi80_lo': 204.0, 'pi80_hi': 2036.0}]
